# LSTM Retraining Bias Simulation Scenarios

This notebook retrains the LSTM from scratch under three bias scenarios.
Unlike the previous simulation notebook (frozen model, test-time only),
here the bias is injected into BOTH training AND test data, then the
model is retrained from scratch on the corrupted training data.

## Three scenarios:
- **Scenario A**: SpO2 systematic downward shift for Black patients (1, 3, 5 points)
- **Scenario B**: Complete temperature removal for Black patients
- **Scenario C**: Heart rate Gaussian noise for Black patients (5, 10, 20 bpm)


## What this answers:
'If this specific bias existed in the real clinical data collection
process from the start, how much would it degrade Black patient
model performance compared to the clean baseline?'

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve
)
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

OUT_DIR  = Path('/home/ino/thesis_outputs')
SIM_DIR  = OUT_DIR / 'simulation_retrain'
SIM_DIR.mkdir(exist_ok=True)

VITAL_NAMES  = ['heart_rate', 'resp_rate', 'spo2', 'temperature', 'map']
VITAL_IDX    = {v: i for i, v in enumerate(VITAL_NAMES)}
KEEP_RACES   = ['White', 'Black', 'Hispanic', 'Asian']
N_FOLDS      = 5
RANDOM_STATE = 42
N_BOOTSTRAP  = 2000

# Training hyperparameters identical to original thesis_cv_full
N_EPOCHS   = 50
BATCH_SIZE = 1024
LR         = 5e-4

# Vital index shortcuts
V_HR   = VITAL_IDX['heart_rate']   # 0
V_SPO2 = VITAL_IDX['spo2']         # 2
V_TEMP = VITAL_IDX['temperature']  # 3

# SpO2 and HR bounds from original notebook
SPO2_MIN, SPO2_MAX = 50.0, 100.0
HR_MIN,   HR_MAX   = 20.0, 300.0

# Scenario configurations
SPO2_SHIFTS   = [0, 1, 3, 5]     # points subtracted from Black SpO2
HR_NOISE_STDS = [0, 5, 10, 20]   # Gaussian noise std in bpm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'Device: {device}  |  GPUs: {n_gpus}')
print(f'Output: {SIM_DIR}')
print(f'Total training runs: {len(SPO2_SHIFTS) + 1 + len(HR_NOISE_STDS)} conditions x {N_FOLDS} folds')

---
## 2. Load & align data

In [ ]:
cv_stay_ids = np.load(OUT_DIR / 'cv_stay_ids.npy')
print(f'CV stay_ids: {len(cv_stay_ids):,}')

cohort_full = pd.read_csv(OUT_DIR / 'cohort.csv')
cohort_full = cohort_full.set_index('stay_id')
cohort_cv   = cohort_full.loc[cv_stay_ids].reset_index()

cohort_full_reset = pd.read_csv(OUT_DIR / 'cohort.csv').reset_index(drop=True)
stay_id_to_ts_idx = {sid: i for i, sid in
                     enumerate(cohort_full_reset['stay_id'].values)}

X_ts_full  = np.load(OUT_DIR / 'X_ts.npy')
M_ts_full  = np.load(OUT_DIR / 'M_ts.npy')
ts_indices = np.array([stay_id_to_ts_idx[sid] for sid in cv_stay_ids])
X_ts_cv    = X_ts_full[ts_indices]
M_ts_cv    = M_ts_full[ts_indices]

keep_mask = cohort_cv['race_group'].isin(KEEP_RACES).values
cohort    = cohort_cv[keep_mask].reset_index(drop=True)
kept_idx  = np.where(keep_mask)[0]

X_ts    = X_ts_cv[kept_idx]   # (N, 24, 5)
M_ts    = M_ts_cv[kept_idx]   # (N, 24, 5)

y        = cohort['hospital_expire_flag'].values.astype(np.int32)
demo     = cohort[['race_group', 'gender', 'age_group']].reset_index(drop=True)
race_arr = demo['race_group'].values

print(f'Cohort: {len(cohort):,}  |  Mortality: {y.mean():.1%}')
print(cohort['race_group'].value_counts().to_string())

# Load baseline OOF probs
oof_baseline = np.load(OUT_DIR / 'oof_probs_LSTM.npy')[kept_idx]
bl_black = average_precision_score(
    y[race_arr=='Black'], oof_baseline[race_arr=='Black'])
bl_white = average_precision_score(
    y[race_arr=='White'], oof_baseline[race_arr=='White'])
print(f'\nBaseline LSTM: Black={bl_black:.4f}  White={bl_white:.4f}  '
      f'Gap={bl_white-bl_black:+.4f}')

---
## 3. Fold splits

In [ ]:
strat_label = (demo['race_group'].astype(str) + '_' +
               cohort['hospital_expire_flag'].astype(str)).values
skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                              random_state=RANDOM_STATE)
fold_splits  = list(skf.split(np.zeros(len(cohort)), strat_label))
print(f'Fold test sizes: {[len(t) for _,t in fold_splits]}')

---
## 4. LSTM model & training infrastructure

In [ ]:
class MortalityLSTM(nn.Module):
    """Identical to thesis_cv_full.ipynb"""
    def __init__(self, input_size=10, hidden_size=128,
                 num_layers=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout, bidirectional=True
        )
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size*2, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last   = self.layer_norm(out[:, -1, :])
        return self.classifier(self.dropout(last)).squeeze(1)


def train_lstm_fold(X_tr_in, y_tr, X_te_in, y_te,
                    n_epochs=N_EPOCHS, lr=LR, fold=0, label=''):
    scale_pw  = torch.tensor(
        [(y_tr==0).sum() / (y_tr==1).sum()], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=scale_pw)

    y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
    y_te_t = torch.tensor(y_te, dtype=torch.float32)

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_tr_in), y_tr_t),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, pin_memory=True)
    val_loader = DataLoader(
        TensorDataset(torch.tensor(X_te_in), y_te_t),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True)

    model = MortalityLSTM()
    if n_gpus > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=3, factor=0.5)

    best_auprc = 0
    best_probs = None

    for epoch in range(1, n_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        probs_list = []
        with torch.no_grad():
            for xb, _ in val_loader:
                probs_list.append(
                    torch.sigmoid(model(xb.to(device))).cpu().numpy())
        probs = np.concatenate(probs_list)
        ap    = average_precision_score(y_te, probs)
        scheduler.step(ap)

        if ap > best_auprc:
            best_auprc = ap
            best_probs = probs.copy()

        if epoch % 10 == 0 or epoch == 1:
            print(f'    [{label}] fold {fold+1} ep {epoch:02d}  '
                  f'AUPRC={ap:.4f}{"  *" if ap==best_auprc else ""}')

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best_probs, best_auprc


def normalise(X_input, X_train):
    ts_mean = X_train.mean(axis=(0,1), keepdims=True)
    ts_std  = X_train.std(axis=(0,1),  keepdims=True) + 1e-8
    return (X_input - ts_mean) / ts_std


def build_input(X_norm, M):
    return np.concatenate([X_norm, M], axis=2).astype(np.float32)


print('LSTM and training helpers ready.')

---
## 5. Bias injection functions

Each function takes the raw X and M arrays for a split and applies
the bias to Black patients only. White patients are never touched.

In [ ]:
def inject_spo2_shift(X, M, race, shift):
    """
    Scenario A: subtract `shift` points from all observed SpO2
    readings for Black patients. Clip to physiological range.
    Clinically motivated by pulse oximetry bias in darker skin.
    """
    if shift == 0:
        return X.copy(), M.copy()
    X_out = X.copy()
    black_loc = np.where(race == 'Black')[0]
    for li in black_loc:
        for h in range(24):
            if M[li, h, V_SPO2] == 1:
                X_out[li, h, V_SPO2] = np.clip(
                    X_out[li, h, V_SPO2] - shift, SPO2_MIN, SPO2_MAX)
    return X_out, M.copy()


def inject_temperature_removal(X, M, race):
    """
    Scenario B: remove ALL temperature measurements for Black patients.
    Replace value with training population median at that hour.
    Note: median is computed per-fold inside the training loop
    to avoid leakage, so we pass the training X here.
    """
    X_out = X.copy()
    M_out = M.copy()
    black_loc = np.where(race == 'Black')[0]
    for li in black_loc:
        for h in range(24):
            if M_out[li, h, V_TEMP] == 1:
                M_out[li, h, V_TEMP] = 0.0
                # Value will be replaced with train median after normalisation
                # For now set to 0 — normalisation will center it anyway
                X_out[li, h, V_TEMP] = 0.0
    return X_out, M_out


def inject_hr_noise(X, M, race, std_bpm, rng):
    """
    Scenario C: add Gaussian noise to observed heart rate readings
    for Black patients. Clip to physiological range.
    Simulates monitoring quality disparities.
    """
    if std_bpm == 0:
        return X.copy(), M.copy()
    X_out = X.copy()
    black_loc = np.where(race == 'Black')[0]
    for li in black_loc:
        for h in range(24):
            if M[li, h, V_HR] == 1:
                noise = rng.normal(0, std_bpm)
                X_out[li, h, V_HR] = np.clip(
                    X_out[li, h, V_HR] + noise, HR_MIN, HR_MAX)
    return X_out, M.copy()


def run_cv(inject_fn, label=''):
    """
    Run full 5-fold CV with a bias injection function.
    inject_fn(X, M, race) -> (X_biased, M_biased)
    Returns out-of-fold probability array.
    """
    oof = np.zeros(len(cohort))
    for fold, (train_idx, test_idx) in enumerate(fold_splits):
        print(f'  [{label}] fold {fold+1}/{N_FOLDS}')

        X_tr = X_ts[train_idx].copy()
        M_tr = M_ts[train_idx].copy()
        X_te = X_ts[test_idx].copy()
        M_te = M_ts[test_idx].copy()
        y_tr = y[train_idx]
        y_te = y[test_idx]
        race_tr = race_arr[train_idx]
        race_te = race_arr[test_idx]

        # Apply bias to BOTH training and test
        X_tr_b, M_tr_b = inject_fn(X_tr, M_tr, race_tr)
        X_te_b, M_te_b = inject_fn(X_te, M_te, race_te)

        # Normalise using biased training statistics
        X_tr_n  = normalise(X_tr_b, X_tr_b)
        X_te_n  = normalise(X_te_b, X_tr_b)
        X_tr_in = build_input(X_tr_n, M_tr_b)
        X_te_in = build_input(X_te_n, M_te_b)

        best_probs, _ = train_lstm_fold(
            X_tr_in, y_tr, X_te_in, y_te,
            fold=fold, label=label)
        oof[test_idx] = best_probs

    return oof


def auprc_by_group(probs):
    """Return dict of AUPRC per race group."""
    out = {'overall': average_precision_score(y, probs)}
    for g in KEEP_RACES:
        mask = race_arr == g
        out[g] = average_precision_score(y[mask], probs[mask])
    out['gap'] = out['White'] - out['Black']
    return out


def bootstrap_delta_p(y_true, p_base, p_new,
                       n_boot=N_BOOTSTRAP, seed=0):
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ys  = y_true[idx]
        if ys.sum() == 0 or ys.sum() == n: continue
        deltas.append(
            average_precision_score(ys, p_new[idx]) -
            average_precision_score(ys, p_base[idx])
        )
    d = np.array(deltas)
    p = 2 * min(np.mean(d <= 0), np.mean(d >= 0))
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5), p


print('Bias injection functions ready.')

---
## 6. Scenario A: SpO2 systematic downward shift

Runs 4 conditions: shift = 0 (clean baseline retrain), 1, 3, 5 points.
Shift=0 gives us a freshly retrained clean baseline to compare against.

In [ ]:
print('=' * 65)
print('SCENARIO A — SpO2 downward shift')
print('=' * 65)

spo2_results = {}

for shift in SPO2_SHIFTS:
    print(f'\nShift = {shift} points')
    fn  = lambda X, M, race, s=shift: inject_spo2_shift(X, M, race, s)
    oof = run_cv(fn, label=f'SpO2-{shift}pt')
    np.save(SIM_DIR / f'oof_spo2_shift_{shift}.npy', oof)
    res = auprc_by_group(oof)
    spo2_results[shift] = {'oof': oof, **res}
    print(f'  Black={res["Black"]:.4f}  White={res["White"]:.4f}  '
          f'Gap={res["gap"]:+.4f}')

print('\nScenario A bootstrap p-values vs clean retrain (shift=0):')
mask_b = race_arr == 'Black'
for shift in SPO2_SHIFTS[1:]:
    mn, lo, hi, p = bootstrap_delta_p(
        y[mask_b],
        spo2_results[0]['oof'][mask_b],
        spo2_results[shift]['oof'][mask_b]
    )
    spo2_results[shift]['p'] = p
    sig = '(significant)' if p < 0.05 else ''
    print(f'  Shift={shift}  delta={mn:+.4f}  [{lo:+.4f},{hi:+.4f}]  '
          f'p={p:.4f}  {sig}')
spo2_results[0]['p'] = 1.0
print('Scenario A complete.')

---
## 7. Scenario B: Complete temperature removal

In [ ]:
print('=' * 65)
print('SCENARIO B — Complete temperature removal for Black patients')
print('=' * 65)

fn_temp = lambda X, M, race: inject_temperature_removal(X, M, race)

print('\nCondition: temperature removed')
oof_temp = run_cv(fn_temp, label='Temp-removed')
np.save(SIM_DIR / 'oof_temperature_removed.npy', oof_temp)
temp_res = auprc_by_group(oof_temp)

# Compare against clean retrain (spo2 shift=0 is our clean baseline)
mn, lo, hi, p = bootstrap_delta_p(
    y[mask_b],
    spo2_results[0]['oof'][mask_b],
    oof_temp[mask_b]
)
temp_res['p'] = p

print(f'  Black={temp_res["Black"]:.4f}  White={temp_res["White"]:.4f}  '
      f'Gap={temp_res["gap"]:+.4f}')
print(f'  delta vs clean baseline={mn:+.4f}  [{lo:+.4f},{hi:+.4f}]  '
      f'p={p:.4f}  {"(significant)" if p < 0.05 else "(not significant)"}')
print('Scenario B complete.')

---
## 8. Scenario C: Heart rate Gaussian noise

In [ ]:
print('=' * 65)
print('SCENARIO C — Heart rate noise injection')
print('=' * 65)

hr_results = {}
rng_hr     = np.random.default_rng(42)

for std in HR_NOISE_STDS:
    print(f'\nNoise std = {std} bpm')
    fn  = lambda X, M, race, s=std: inject_hr_noise(X, M, race, s, rng_hr)
    oof = run_cv(fn, label=f'HR-{std}bpm')
    np.save(SIM_DIR / f'oof_hr_noise_{std}.npy', oof)
    res = auprc_by_group(oof)
    hr_results[std] = {'oof': oof, **res}
    print(f'  Black={res["Black"]:.4f}  White={res["White"]:.4f}  '
          f'Gap={res["gap"]:+.4f}')

print('\nScenario C bootstrap p-values vs clean retrain (std=0):')
for std in HR_NOISE_STDS[1:]:
    mn, lo, hi, p = bootstrap_delta_p(
        y[mask_b],
        hr_results[0]['oof'][mask_b],
        hr_results[std]['oof'][mask_b]
    )
    hr_results[std]['p'] = p
    sig = '(significant)' if p < 0.05 else ''
    print(f'  Std={std:2d}  delta={mn:+.4f}  [{lo:+.4f},{hi:+.4f}]  '
          f'p={p:.4f}  {sig}')
hr_results[0]['p'] = 1.0
print('Scenario C complete.')

---
## 9. Figures

In [ ]:
# Clean retrained baseline (shift=0) for comparison in all figures
clean_bl_black = spo2_results[0]['Black']
clean_bl_white = spo2_results[0]['White']
clean_bl_gap   = spo2_results[0]['gap']

print(f'Clean retrained baseline:')
print(f'  Black={clean_bl_black:.4f}  White={clean_bl_white:.4f}  '
      f'Gap={clean_bl_gap:+.4f}')
print(f'  (Original frozen-model baseline: '
      f'Black={bl_black:.4f}  White={bl_white:.4f})')

In [ ]:
# Figure 1: Scenario A — SpO2 sensitivity curve 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

shifts      = SPO2_SHIFTS
black_auprc = [spo2_results[s]['Black'] for s in shifts]
white_auprc = [spo2_results[s]['White'] for s in shifts]

ax = axes[0]
ax.plot(shifts, black_auprc, 'o-', color='#D65F5F', linewidth=2.5,
        markersize=8, label='Black patients (shifted)')
ax.axhline(clean_bl_white, color='#4878CF', linestyle='--', linewidth=1.5,
           label=f'White baseline ({clean_bl_white:.3f})')
ax.axhline(clean_bl_black, color='#D65F5F', linestyle=':', linewidth=1.5,
           label=f'Black clean baseline ({clean_bl_black:.3f})')
for s in shifts[1:]:
    if spo2_results[s].get('p', 1) < 0.05:
        ax.plot(s, spo2_results[s]['Black'], '*', color='black',
                markersize=14, zorder=5)
ax.set_xlabel('SpO2 shift (points subtracted from Black patients)')
ax.set_ylabel('AUPRC')
ax.set_title('Scenario A — SpO2 bias (retrained model)\n'
             '* = significant vs clean baseline p<0.05')
ax.legend(fontsize=9); ax.set_xticks(shifts)

ax = axes[1]
gaps = [spo2_results[s]['gap'] for s in shifts]
ax.bar(shifts, gaps, color='#c0392b', alpha=0.75, edgecolor='white')
ax.axhline(clean_bl_gap, color='gray', linestyle='--', linewidth=1.5,
           label=f'Clean baseline gap ({clean_bl_gap:.3f})')
ax.set_xlabel('SpO2 shift (points)')
ax.set_ylabel('White AUPRC − Black AUPRC')
ax.set_title('Growing disparity gap as SpO2 bias increases\n(retrained model)')
ax.legend(fontsize=9); ax.set_xticks(shifts)

plt.suptitle('Scenario A: SpO2 systematic bias — LSTM retrained under bias',
             fontsize=13)
plt.tight_layout()
plt.savefig(SIM_DIR / 'fig_retrain_scenario_a_spo2.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_retrain_scenario_a_spo2.png')

In [ ]:
# Figure 2: Scenario B — Temperature removal
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
vals   = [clean_bl_black, temp_res['Black']]
labels = ['Clean\nbaseline', 'Temperature\nremoved']
colors = ['#7f8c8d', '#8e44ad']
bars   = ax.bar(labels, vals, color=colors, edgecolor='white',
                width=0.5, alpha=0.85)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.003,
            f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.axhline(clean_bl_white, color='#4878CF', linestyle='--', linewidth=1.5,
           label=f'White baseline ({clean_bl_white:.3f})')
ax.set_ylabel('AUPRC')
ax.set_title('Scenario B — Temperature removed\nBlack patients only (retrained)')
ax.text(0.5, 0.05,
        f'delta={temp_res["Black"]-clean_bl_black:+.4f}  '
        f'p={temp_res["p"]:.3f}',
        ha='center', transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3',
                  facecolor='lightyellow', alpha=0.8))
ax.set_ylim(0, max(vals + [clean_bl_white])*1.15)
ax.legend(fontsize=9)

ax = axes[1]
yb = y[mask_b]
prec_bl, rec_bl, _ = precision_recall_curve(
    yb, spo2_results[0]['oof'][mask_b])
prec_te, rec_te, _ = precision_recall_curve(yb, oof_temp[mask_b])
ax.plot(rec_bl, prec_bl, color='#7f8c8d', linewidth=2,
        label=f'Clean baseline (AUPRC={clean_bl_black:.3f})')
ax.plot(rec_te, prec_te, color='#8e44ad', linewidth=2, linestyle='--',
        label=f'Temp removed (AUPRC={temp_res["Black"]:.3f})')
ax.axhline(yb.mean(), color='gray', linestyle=':', alpha=0.6,
           label=f'Prevalence ({yb.mean():.2f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR curves — Black patients')
ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1)

plt.suptitle('Scenario B: Complete temperature removal — retrained LSTM',
             fontsize=13)
plt.tight_layout()
plt.savefig(SIM_DIR / 'fig_retrain_scenario_b_temperature.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_retrain_scenario_b_temperature.png')

In [ ]:
# Figure 3: Scenario C — Heart rate noise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

noise_levels = HR_NOISE_STDS
hr_black     = [hr_results[s]['Black'] for s in noise_levels]

ax = axes[0]
ax.plot(noise_levels, hr_black, 'o-', color='#D65F5F', linewidth=2.5,
        markersize=8, label='Black patients (noise added)')
ax.axhline(clean_bl_white, color='#4878CF', linestyle='--', linewidth=1.5,
           label=f'White baseline ({clean_bl_white:.3f})')
ax.axhline(clean_bl_black, color='#D65F5F', linestyle=':', linewidth=1.5,
           label=f'Black clean baseline ({clean_bl_black:.3f})')
for s in noise_levels[1:]:
    if hr_results[s].get('p', 1) < 0.05:
        ax.plot(s, hr_results[s]['Black'], '*', color='black',
                markersize=14, zorder=5)
ax.set_xlabel('Heart rate noise std (bpm)')
ax.set_ylabel('AUPRC')
ax.set_title('Scenario C — Heart rate noise (retrained model)\n'
             '* = significant vs clean baseline p<0.05')
ax.legend(fontsize=9); ax.set_xticks(noise_levels)

ax = axes[1]
gaps = [hr_results[s]['gap'] for s in noise_levels]
ax.bar(noise_levels, gaps, color='#e67e22', alpha=0.75, edgecolor='white')
ax.axhline(clean_bl_gap, color='gray', linestyle='--', linewidth=1.5,
           label=f'Clean baseline gap ({clean_bl_gap:.3f})')
ax.set_xlabel('Heart rate noise std (bpm)')
ax.set_ylabel('White AUPRC − Black AUPRC')
ax.set_title('Growing disparity gap as noise increases\n(retrained model)')
ax.legend(fontsize=9); ax.set_xticks(noise_levels)

plt.suptitle('Scenario C: Heart rate noise bias — LSTM retrained under bias',
             fontsize=13)
plt.tight_layout()
plt.savefig(SIM_DIR / 'fig_retrain_scenario_c_hr_noise.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_retrain_scenario_c_hr_noise.png')

In [ ]:
# Figure 4: Frozen vs retrained comparison 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scenario A comparison
ax = axes[0]
shifts_plot = [1, 3, 5]
frozen_bl   = [bl_black - (bl_black - spo2_results[s]['Black'])
               for s in shifts_plot]  # approximate from previous run
retrain_bl  = [spo2_results[s]['Black'] for s in shifts_plot]
x = np.arange(len(shifts_plot))
ax.bar(x - 0.2, [bl_black]*3, 0.35, label='Frozen baseline',
       color='#95a5a6', alpha=0.7)
ax.bar(x + 0.2, retrain_bl, 0.35, label='Retrained under bias',
       color='#c0392b', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f'SpO2\n-{s}pt' for s in shifts_plot])
ax.set_ylabel('Black AUPRC')
ax.set_title('Scenario A — SpO2 shift')
ax.legend(fontsize=9)
ax.axhline(clean_bl_white, color='#4878CF', linestyle='--',
           linewidth=1, alpha=0.6, label='White')

# Scenario B comparison
ax = axes[1]
ax.bar([0, 1], [bl_black, temp_res['Black']],
       color=['#95a5a6', '#8e44ad'], edgecolor='white',
       width=0.5, alpha=0.85)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Frozen\nbaseline', 'Retrained\n(temp removed)'])
ax.set_ylabel('Black AUPRC')
ax.set_title('Scenario B — Temperature removal')
for i, val in enumerate([bl_black, temp_res['Black']]):
    ax.text(i, val+0.003, f'{val:.4f}', ha='center',
            fontsize=10, fontweight='bold')
ax.axhline(clean_bl_white, color='#4878CF', linestyle='--',
           linewidth=1, alpha=0.6)

# Scenario C comparison
ax = axes[2]
noise_plot  = [5, 10, 20]
retrain_hr  = [hr_results[s]['Black'] for s in noise_plot]
x = np.arange(len(noise_plot))
ax.bar(x - 0.2, [bl_black]*3, 0.35, label='Frozen baseline',
       color='#95a5a6', alpha=0.7)
ax.bar(x + 0.2, retrain_hr, 0.35, label='Retrained under bias',
       color='#e67e22', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f'HR\n±{s}bpm' for s in noise_plot])
ax.set_ylabel('Black AUPRC')
ax.set_title('Scenario C — Heart rate noise')
ax.legend(fontsize=9)
ax.axhline(clean_bl_white, color='#4878CF', linestyle='--',
           linewidth=1, alpha=0.6)

plt.suptitle('Black patient AUPRC — frozen baseline vs retrained under bias\n'
             'Retrained models show true long-term effect of systematic bias',
             fontsize=12)
plt.tight_layout()
plt.savefig(SIM_DIR / 'fig_retrain_vs_frozen_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_retrain_vs_frozen_comparison.png')

In [ ]:
# Figure 5: Overview of all retrained scenarios
fig, ax = plt.subplots(figsize=(14, 6))

conditions = ['Clean\nbaseline']
black_vals = [clean_bl_black]
bar_colors = ['#7f8c8d']

conditions.append('Temp\nremoved')
black_vals.append(temp_res['Black'])
bar_colors.append('#8e44ad')

for s in [1, 3, 5]:
    conditions.append(f'SpO2\n-{s}pt')
    black_vals.append(spo2_results[s]['Black'])
    bar_colors.append('#c0392b')

for s in [5, 10, 20]:
    conditions.append(f'HR\n±{s}bpm')
    black_vals.append(hr_results[s]['Black'])
    bar_colors.append('#e67e22')

x    = np.arange(len(conditions))
bars = ax.bar(x, black_vals, color=bar_colors, edgecolor='white', alpha=0.85)
ax.axhline(clean_bl_black, color='#D65F5F', linestyle='--', linewidth=1.5,
           label=f'Black clean baseline ({clean_bl_black:.3f})')
ax.axhline(clean_bl_white, color='#4878CF', linestyle='--', linewidth=1.5,
           label=f'White baseline ({clean_bl_white:.3f})')

for bar, val in zip(bars, black_vals):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.003,
            f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')

import matplotlib.patches as mpatches
legend_patches = [
    mpatches.Patch(color='#7f8c8d', label='Clean baseline (retrained)'),
    mpatches.Patch(color='#8e44ad', label='Temperature removed'),
    mpatches.Patch(color='#c0392b', label='SpO2 shift'),
    mpatches.Patch(color='#e67e22', label='Heart rate noise'),
]
ax.set_xticks(x)
ax.set_xticklabels(conditions, fontsize=9)
ax.set_ylabel('AUPRC — Black patients')
ax.set_title('Black patient AUPRC — all retrained bias scenarios\n'
             'LSTM retrained from scratch under each bias condition', fontsize=12)
ax.legend(handles=legend_patches + [
    plt.Line2D([0],[0], color='#D65F5F', linestyle='--', label=f'Black baseline ({clean_bl_black:.3f})'),
    plt.Line2D([0],[0], color='#4878CF', linestyle='--', label=f'White baseline ({clean_bl_white:.3f})'),
], fontsize=9, loc='lower left')

plt.tight_layout()
plt.savefig(SIM_DIR / 'fig_retrain_all_scenarios_overview.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_retrain_all_scenarios_overview.png')

---
## 10. Summary table

In [ ]:
summary_rows = []

summary_rows.append({
    'scenario': 'Clean baseline (retrained)',
    'condition': 'shift=0',
    'black_auprc': clean_bl_black,
    'white_auprc': clean_bl_white,
    'gap': clean_bl_gap,
    'delta_black': 0.0,
    'p_value': None
})

summary_rows.append({
    'scenario': 'Scenario B — Temperature removed',
    'condition': 'all temp removed',
    'black_auprc': temp_res['Black'],
    'white_auprc': temp_res['White'],
    'gap': temp_res['gap'],
    'delta_black': temp_res['Black'] - clean_bl_black,
    'p_value': temp_res['p']
})

for s in SPO2_SHIFTS[1:]:
    summary_rows.append({
        'scenario': f'Scenario A — SpO2 shift',
        'condition': f'-{s}pt',
        'black_auprc': spo2_results[s]['Black'],
        'white_auprc': spo2_results[s]['White'],
        'gap': spo2_results[s]['gap'],
        'delta_black': spo2_results[s]['Black'] - clean_bl_black,
        'p_value': spo2_results[s].get('p', None)
    })

for s in HR_NOISE_STDS[1:]:
    summary_rows.append({
        'scenario': f'Scenario C — HR noise',
        'condition': f'std={s}bpm',
        'black_auprc': hr_results[s]['Black'],
        'white_auprc': hr_results[s]['White'],
        'gap': hr_results[s]['gap'],
        'delta_black': hr_results[s]['Black'] - clean_bl_black,
        'p_value': hr_results[s].get('p', None)
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SIM_DIR / 'simulation_retrain_summary.csv', index=False)

print('=' * 75)
print('RETRAINED SIMULATION SUMMARY')
print('=' * 75)
print(summary_df[['scenario','condition','black_auprc',
                  'white_auprc','gap','delta_black','p_value']].to_string(index=False))

print(f'\nAll outputs saved to: {SIM_DIR}')
for f in sorted(SIM_DIR.glob('*')):
    print(f'  {f.name}')